In [5]:
import sys
sys.path.append("..")

import pandas as pd
from src.config import LOG_DIR, CV_DIR
from src.experiment_log import export_excel

# Lọc bỏ các dòng bị ghi nhầm "tuan" ở cột Ten mo hinh trong log_tuan.csv
tuan_log_path = LOG_DIR / "log_tuan.csv"
t = pd.read_csv(tuan_log_path)
t = t[t["Ten mo hinh"] != "tuan"]
t.to_csv(tuan_log_path, index=False)

# Xuất file Excel tổng hợp cho cả nhóm
export_excel()

# Đọc và gộp tất cả các file log_*.csv
log_files = sorted(LOG_DIR.glob("log_*.csv"))
log_dfs = []
for f in log_files:
    current_df = pd.read_csv(f)
    log_dfs.append(current_df)

df = pd.concat(log_dfs, ignore_index=True)

# Loại bỏ dòng trùng theo cặp (Nguoi chay, Ten mo hinh), giữ lần chạy mới nhất
df_dedup = df.drop_duplicates(subset=["Nguoi chay", "Ten mo hinh"], keep="last")

# Bỏ phần chú thích trong ngoặc ở cuối (nếu có) rồi chuẩn hóa tên chiến lược
cleaned_strategies = []
for val in df_dedup["Chien luoc xu ly mat can bang"]:
    s = str(val)
    paren_idx = s.find("(")
    if paren_idx != -1:
        s = s[:paren_idx].rstrip()
    cleaned_strategies.append(s)

chuan = {
    "Khong xu ly": "N0 khong xu ly",
    "N0: khong xu ly": "N0 khong xu ly",
    "class_weight=None": "N0 khong xu ly",
    "N1: class_weight=balanced": "class_weight=balanced",
    "N2_under": "N2 undersampling",
    "N3_smote": "N3 SMOTE",
    "N4_under_smote": "N4 undersampling + SMOTE",
}
df_dedup["Chien luoc"] = pd.Series(cleaned_strategies, index=df_dedup.index).replace(chuan)

# Chọn cột, sắp xếp theo Macro-F1 giảm dần và lưu ra summary_all.csv
cols = [
    "Nguoi chay",
    "Ten mo hinh",
    "Chien luoc",
    "Macro-F1 (TB)",
    "Macro-F1 (std)",
    "Balanced Acc (TB)",
    "Recall c4 Metasploit_SSH",
    "Recall c5 NMAP_FIN_SCAN",
    "Recall c11 Wipro_bulb",
    "Fit time (s)",
]
summary_df = df_dedup[cols]
summary_sorted = summary_df.sort_values(by="Macro-F1 (TB)", ascending=False)
summary = summary_sorted.round(4)

output_csv = CV_DIR / "summary_all.csv"
summary.to_csv(output_csv, index=False)

summary

,Nguoi chay,Ten mo hinh,Chien luoc,Macro-F1 (TB),Macro-F1 (std),Balanced Acc (TB),Recall c4 Metasploit_SSH,Recall c5 NMAP_FIN_SCAN,Recall c11 Wipro_bulb,Fit time (s)
13,bao,hgbT_N2_under,N2 undersampling,0.9688,0.0126,0.9565,0.7600,0.82,0.9429,10.2175
77,tuan,hgb_N1_balanced_rep3,CV lap 3x5,0.9661,0.0143,0.9558,0.7222,0.82,0.9657,50.5291
1,Khanh,final_bo_proto_service,class_weight=balanced,0.9656,0.0124,0.9558,0.7267,0.82,0.9600,38.0531
9,Khanh,rf_bo_nhom_active_idle,N0 khong xu ly,0.9649,0.0127,0.9527,0.7600,0.82,0.9143,2.0436
40,thai,rf_N2_under,N2 undersampling,0.9645,0.0142,0.9526,0.7600,0.82,0.9143,0.9627
78,tuan,xgb_default_rep3,CV lap 3x5,0.9644,0.0175,0.9568,0.7578,0.82,0.9600,23.6387
5,Khanh,rf_bo_nhom_kich_thuoc_header,N0 khong xu ly,0.9637,0.0140,0.9510,0.7600,0.82,0.9029,2.2571
39,thai,hgb_N1_balanced,class_weight=balanced,0.9633,0.0127,0.9561,0.7267,0.82,0.9657,36.7680
73,tuan,rf_d10_cw_None,N0 khong xu ly,0.9630,0.0131,0.9503,0.7600,0.82,0.8971,1.2863
72,tuan,random_forest_tuned,N0 khong xu ly,0.9630,0.0131,0.9503,0.7600,0.82,0.8971,1.3983


In [6]:
import pandas as pd
from scipy import stats
from src.config import CV_DIR

# Danh sách các mô hình ứng viên; phần tử đầu tiên làm mốc so sánh
ung_vien = [
    "hgbT_N2_under",
    "rf_N2_under",
    "random_forest_tuned",
    "xgb_default",
    "hgb_tuned",
]

model_goc = ung_vien[0]
goc_df = pd.read_csv(CV_DIR / f"{model_goc}.csv")
goc_scores = goc_df["macro_f1"]

comparison_rows = []
col_diff_name = "chenh_so_voi_" + model_goc

# So sánh từng mô hình ứng viên với mô hình gốc theo từng fold
for candidate_name in ung_vien[1:]:
    cand_df = pd.read_csv(CV_DIR / f"{candidate_name}.csv")
    cand_scores = cand_df["macro_f1"]

    # Tính chênh lệch trung bình và chạy kiểm định t ghép cặp
    diff_series = cand_scores - goc_scores
    mean_diff = round(float(diff_series.mean()), 4)

    t_stat, p_val = stats.ttest_rel(cand_scores, goc_scores)
    t_rounded = round(float(t_stat), 2)
    p_rounded = round(float(p_val), 3)

    row_item = {
        "mo_hinh": candidate_name,
        col_diff_name: mean_diff,
        "t": t_rounded,
        "p": p_rounded,
    }
    comparison_rows.append(row_item)

result_df = pd.DataFrame(comparison_rows)
result_df

,mo_hinh,chenh_so_voi_hgbT_N2_under,t,p
0,rf_N2_under,-0.0043,-1.49,0.212
1,random_forest_tuned,-0.0058,-2.19,0.094
2,xgb_default,-0.0074,-1.18,0.305
3,hgb_tuned,-0.0087,-1.94,0.124


In [7]:
import numpy as np
import pandas as pd
from scipy import stats
from src.config import CV_DIR

# Danh sách mô hình cần so sánh; phần tử đầu tiên làm mốc đối chiếu
ung_vien = [
    "hgbT_N2_under",
    "rf_N2_under",
    "hgb_N1_balanced",
    "random_forest_tuned",
    "random_forest",
    "xgb_default",
]

model_goc = ung_vien[0]
goc_df = pd.read_csv(CV_DIR / f"{model_goc}.csv")
goc_scores = goc_df["macro_f1"].to_numpy()

results = []

# Duyệt qua từng mô hình ứng viên để so sánh với mô hình gốc
for candidate_name in ung_vien[1:]:
    cand_df = pd.read_csv(CV_DIR / f"{candidate_name}.csv")
    cand_scores = cand_df["macro_f1"].to_numpy()

    # Độ chênh điểm từng fold và số lượng fold
    diff = cand_scores - goc_scores
    k = len(diff)
    mean_diff = float(diff.mean())
    var_diff = float(diff.var(ddof=1))

    # Công thức hiệu chỉnh Nadeau & Bengio cho các fold có chung dữ liệu huấn luyện
    adjusted_var = (1.0 / k + 0.25) * var_diff
    if adjusted_var > 0 and k > 1:
        t_hc = mean_diff / np.sqrt(adjusted_var)
        # Tính p-value hai phía từ phân phối t với bậc tự do k - 1
        p_hc = 2.0 * stats.t.sf(abs(t_hc), df=k - 1)
    else:
        t_hc = 0.0
        p_hc = 1.0

    row = {
        "mo_hinh": candidate_name,
        "chenh": round(mean_diff, 4),
        "t_hieu_chinh": round(float(t_hc), 2),
        "p_hieu_chinh": round(float(p_hc), 3),
    }
    results.append(row)

summary_table = pd.DataFrame(results)
summary_table

,mo_hinh,chenh,t_hieu_chinh,p_hieu_chinh
0,rf_N2_under,-0.0043,-0.99,0.378
1,hgb_N1_balanced,-0.0055,-1.13,0.321
2,random_forest_tuned,-0.0058,-1.46,0.218
3,random_forest,-0.0070,-1.81,0.145
4,xgb_default,-0.0074,-0.78,0.477
